In [0]:
from pyspark.sql.functions import col
print("Welcome to the W261 final project Weather Data EDA!") 


# Mount the Data

In [0]:
data_BASE_DIR = "dbfs:/mnt/mids-w261/"
display(dbutils.fs.ls(f"{data_BASE_DIR}"))

# Weather Data



In [0]:
# Weather data
df_weather = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_3m/")
display(df_weather)

   
# Sample the Data
My approach is to sample the data and convert it to pandas to utilize the pandas library direclty. At the end, I will convert the clean data to parquet format.

In [0]:
# convert to pandas
sample_weather= df_weather.sample(withReplacement=False, fraction=0.01, seed=1)
weather_pd = sample_weather.toPandas()
weather_pd.head(10)

# Missing Value Handling

In [0]:
# number of records
df_weather.count()

In [0]:
# number of columns
len(weather_pd.columns)

In [0]:
# number of columns that have more than 50% null values
null_counts = weather_pd.isnull().sum()
null_percent = ((null_counts / len(weather_pd)) * 100).sort_values(ascending=False)
num_cols_over_50_null = (null_percent > 50).sum()
display(num_cols_over_50_null)

In [0]:
# quick look to see which columns have more than 50% null values
null_info = null_percent.sort_values(ascending=False)
display(null_info)


Weather data has 124 columns 30528602 rows.

STATION, DATE, SOURCE, REPORT_TYPE, YEAR columns have 0 missing data.

In the weather dataset, 105 out of 124 clumns (approximately 85%) have more than 50% null values. Therefore, these columns will be excluded from this EDA.

In [0]:
# columns to drop
c_drop = null_percent[null_percent > 50].index
weather_clean = weather_pd.drop(c_drop, axis=1)

# Drop YEAR (redundant with DATE)
weather_clean = weather_clean.drop(columns=['YEAR'])

# Drop REM ()
len(weather_clean.columns)

# Data Type Conversion & Column Renaming

In [0]:
# quick look to dtypes of each column we have in the final version of the dataset
weather_clean.info()

In [0]:
display(weather_clean)

#STATION: (Unique weather station identifier, e.g. WBAN or ICAO code) -> String

#DATE: (Timestamp of the weather observation) -> Datetime

#LATITUDE: (Station's north-south position in decimal degrees) -> Numeric

#LONGITUDE: (Station's east-west position in decimal degrees) -> Numeric

#ELEVATION: (Station's vertical distance above sea level in meters) -> Numeric

#NAME: (Station name with city and country/state) -> String

#REPORT_TYPE: (Type of weather observation, e.g. FM-15 for METAR, FM-16 for SPECI) -> Categorical

#SOURCE: (Numeric code indicating the data source/provider) -> Categorical

#HourlyAltimeterSetting: (Atmospheric pressure adjusted to sea level, in inches of mercury) -> Numeric

#HourlyDewPointTemperature: (Temperature at which air becomes saturated, in °F) -> Numeric

#HourlyDryBulbTemperature: (Ambient air temperature measured by a standard thermometer, in °F) -> Numeric

#HourlyRelativeHumidity: (Percentage of moisture in the air relative to saturation) -> Numeric

#HourlySkyConditions: (Cloud cover description, e.g. FEW, SCT, BKN, OVC with altitude) -> String

#HourlyStationPressure: (Atmospheric pressure at the station's elevation, in inches of mercury) -> Numeric

#HourlyVisibility: (Horizontal distance at which objects can be seen, in statute miles) -> Numeric

#HourlyWindDirection: (Compass direction the wind is blowing from, in degrees 0-360) -> Numeric

#HourlyWindSpeed: (Speed of sustained wind, in miles per hour) -> Numeric

#REM: (Raw alphanumeric remarks text transmitted by the weather station) -> String

In [0]:
# rename columns & change dtypes
import pandas as pd

# convert DATE to datetime
weather_clean['DATE'] = pd.to_datetime(weather_clean['DATE'], errors='coerce')

# numeric columns
numeric_cols = [
    'LATITUDE', 'LONGITUDE', 'ELEVATION',
    'HourlyAltimeterSetting', 'HourlyDewPointTemperature',
    'HourlyDryBulbTemperature', 'HourlyRelativeHumidity',
    'HourlyStationPressure', 'HourlyVisibility',
    'HourlyWindDirection', 'HourlyWindSpeed'
]
for col in numeric_cols:
    weather_clean[col] = pd.to_numeric(weather_clean[col], errors='coerce')

# categorical columns
weather_clean['REPORT_TYPE'] = weather_clean['REPORT_TYPE'].astype('category')
weather_clean['SOURCE'] = weather_clean['SOURCE'].astype('category')

# Keep as string (already object): STATION, NAME, HourlySkyConditions, REM 

# rename columns to snake_case for clarity 
weather_clean = weather_clean.rename(columns={
    'STATION': 'station',
    'DATE': 'date',
    'LATITUDE': 'latitude',
    'LONGITUDE': 'longitude',
    'ELEVATION': 'elevation',
    'NAME': 'name',
    'REPORT_TYPE': 'report_type',
    'SOURCE': 'source',
    'HourlyAltimeterSetting': 'hourly_altimeter_setting',
    'HourlyDewPointTemperature': 'hourly_dew_point_temp',
    'HourlyDryBulbTemperature': 'hourly_dry_bulb_temp',
    'HourlyRelativeHumidity': 'hourly_relative_humidity',
    'HourlySkyConditions': 'hourly_sky_conditions',
    'HourlyStationPressure': 'hourly_station_pressure',
    'HourlyVisibility': 'hourly_visibility',
    'HourlyWindDirection': 'hourly_wind_direction',
    'HourlyWindSpeed': 'hourly_wind_speed',
    'REM': 'rem'
})


print(weather_clean.dtypes)
print(f"\nShape: {weather_clean.shape}")

# Duplicated Records

In [0]:
# check duplicated records 
duplicate_count = weather_clean.duplicated().sum()
print(duplicate_count)

# Outlier Detection 


In [0]:
# IQR-based outlier detection for each numeric column
outlier_summary = []
num_cols = [c for c in weather_clean.select_dtypes(include='number').columns if c not in ['latitude', 'longitude','elevation','hourly_station_pressure']]

for col in num_cols:
    series = weather_clean[col].dropna()
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((series < lower) | (series > upper)).sum()
    pct_outliers = round(n_outliers / len(series) * 100, 2)

    outlier_summary.append({
        'column': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'lower_bound': round(lower, 2),
        'upper_bound': round(upper, 2),
        'n_outliers': n_outliers,
        'pct_outliers': pct_outliers,
        'total_non_null': len(series)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('pct_outliers', ascending=False)
display(outlier_df)

# Cardinality Detection

In [0]:
car_cols = ["station","name","report_type","source","hourly_sky_conditions","rem"]

for i in car_cols:
    score = round(weather_clean[i].nunique() / len(weather_clean), 5)
    print(f"Cardinality score of column {i} :{score}")

# Colum Statistics After Cleaning

In [0]:

# numeric columns after renaming (exclude latitude longitude since they are just location indicators)
num_cols = [c for c in weather_clean.select_dtypes(include='number').columns if c not in ['latitude', 'longitude']]

# describe
desc = weather_clean[num_cols].describe().T

# median and mode
desc['median'] = weather_clean[num_cols].median()
desc['mode'] = weather_clean[num_cols].mode().iloc[0]

# reorder columns for readability
desc = desc[['count', 'mean', 'median', 'mode', 'std', 'min', '25%', '50%', '75%', 'max']]

# make column names visible as a column
desc = desc.reset_index().rename(columns={'index': 'column'})

display(desc)

#Histograms and Box Plots

In [0]:
import matplotlib.pyplot as plt
import math

# exclude latitude/longitude (just location indicators, not weather measurements)
plot_cols = [c for c in num_cols if c not in ['latitude', 'longitude']]

# plot histograms + boxplots side by side for each numeric column
n = len(plot_cols)
fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n))

for i, col in enumerate(plot_cols):
    data = weather_clean[col].dropna()

    # histogram
    axes[i, 0].hist(data, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[i, 0].set_title(f'{col} — Histogram')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Frequency')

    # boxplot
    axes[i, 1].boxplot(data, vert=False, patch_artist=True,
                        boxprops=dict(facecolor='steelblue', alpha=0.7))
    axes[i, 1].set_title(f'{col} — Boxplot')
    axes[i, 1].set_xlabel(col)

plt.tight_layout()
plt.show()

#Correlation Matrix


In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

correlation_matrix = weather_clean.corr(numeric_only=True)
print("Correlation Matrix:")
print(correlation_matrix)

# 3. Visualize the correlation matrix as a heatmap (optional, but recommended)
plt.figure(figsize=(8, 6)) # Adjust figure size
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix Heatmap')
plt.show()

In [0]:
# OTPW
df_otpw = spark.read.format("csv").option("header","true").load(f"dbfs:/mnt/mids-w261/OTPW_3M_2015.csv")
display(df_otpw)

## Where to Checkpoint your data?
There is a folder created in the Mount called `student-groups`. Please create a folder there with the name `Group_{section}_{number}` for example, Group Section 01 Number 01 will be `Group_01_01`. Any folder that doesn't follow this convection will be deleted without warning. Thanks! 

In [0]:
# Create folder
section = "3"
number = "1"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"
dbutils.fs.mkdirs(folder_path)

# Save df_weather as a parquet file
spark.createDataFrame(weather_clean).write.mode('overwrite').parquet(f"{folder_path}/weather_clean.parquet")

# Pipeline Steps For Classification Problem

These are the "normal" steps for a Classification Pipeline! Of course, you can try more!

## 1. Data cleaning and preprocessing

* Remove outliers or missing values
* Encode categorical features
* Scale numerical features

## 2. Feature selection

* Select the most important features for the model
* Use univariate feature selection, recursive feature elimination, or random forest feature importance

## 3. Model training

* Train a machine learning model to predict delays more than 15 minutes
* Use logistic regression, decision trees, random forests, or support vector machines

## 4. Model evaluation

* Evaluate the performance of the trained model on a holdout dataset
* Use accuracy, precision, recall, or F1 score

## 5. Model deployment

* Deploy the trained model to a production environment
* Deploy the model as a web service or as a mobile app

## Tools

* Spark's MLlib and SparkML libraries
* These libraries have parallelized methods for data cleaning and preprocessing, feature selection, model training, model evaluation, and model deployment which we will utilize for this classification problem.
